# Low-Rank Class-Imbalance Decomposition (Experimental)

An exploratory classification head that decomposes each layer's weights into a shared "general" component and a low-rank, task-adapted correction, to study how much extra capacity minority classes need relative to majority classes. Trained with a class-balanced weighted sampler, a margin loss, a supervised-contrastive term, and a sinusoidally-annealed discrepancy loss between the full and general-only outputs. This is a research probe, not the primary reported model — see the main README for the headline SVM / quadruplet-embedding results. Cleaned, importable version: `src/models/low_rank_decomposition.py`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load extracted deep features

In [ ]:
X=np.load("features.npy")
y= np.load("labels.npy")

In [ ]:


# mask for removing class 8
mask = y_test != 8

X_test = X_test[mask]
y_test = y_test[mask]

In [ ]:
print("Feature shape:", X.shape)
print("Label shape:", y.shape)


In [ ]:
print("Feature shape:", X_test.shape)
print("Label shape:", y_test.shape)

## 2. Class distribution

In [ ]:
import numpy as np

# Load the files
features = np.load('features.npy')
labels = np.load('labels.npy')

# Get unique values and their corresponding counts
classes, counts = np.unique(labels, return_counts=True)

# Print the results
print(f"{'Class/Value':<15} | {'Count':<10}")
print("-" * 30)
for cls, count in zip(classes, counts):
    print(f"{cls:<15} | {count:<10}")

# Quick check on total rows
print(f"\nTotal rows: {features.shape[0]}")

## 3. Weighted sampling
Inverse-frequency sample weights so minority classes are seen roughly as often as majority classes during training.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np

# 🔥 Ensure y is numpy (important)
y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else y

# =========================
# Dataset
# =========================
class NPYDataset(Dataset):
    def __init__(self, X, y):
        # avoid double conversion
        self.X = torch.tensor(X, dtype=torch.float32) if not isinstance(X, torch.Tensor) else X
        self.y = torch.tensor(y, dtype=torch.long) if not isinstance(y, torch.Tensor) else y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = NPYDataset(X, y_np)

# =========================
# Weighted Sampler
# =========================

# count samples per class
class_counts = np.bincount(y_np)

# avoid division by zero
class_weights = 1.0 / (class_counts + 1e-6)

# assign weight to each sample
sample_weights = class_weights[y_np]

# convert to torch
sample_weights = torch.tensor(sample_weights, dtype=torch.float32)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# =========================
# DataLoader
# =========================
loader = DataLoader(
    dataset,
    batch_size=64,
    sampler=sampler   # 🔥 IMPORTANT: no shuffle here
)

print("✅ DataLoader with WeightedRandomSampler ready!")

In [ ]:
class NPYDataset1(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

test_dataset = NPYDataset1(X_test, y_test)

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 4. Low-rank decomposed linear layer
`y = Wg(x) + x @ (B @ A)^T` — a shared general weight matrix plus a low-rank, task-adapted correction. See `src/models/low_rank_decomposition.py::LowRankLinear`.

In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=32):
        super().__init__()
        
        # General weights
        self.Wg = nn.Linear(in_features, out_features)
        
        # Low-rank decomposition
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.randn(out_features, rank) * 0.01)

    def forward(self, x):
        Wt = self.B @ self.A   # low-rank matrix
        return self.Wg(x) + x @ Wt.T
        
    def forward_general(self, x):
        return self.Wg(x)

## 5. Classifier built from low-rank layers

In [ ]:
class Model(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        self.layer1 = LowRankLinear(input_dim, 128)
        self.relu = nn.ReLU()
        self.layer2 = LowRankLinear(128, num_classes)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

    def forward_general(self, x):
        x = self.relu(self.layer1.forward_general(x))
        x = self.layer2.forward_general(x)
        return x

## 6. Supervised contrastive loss (auxiliary term)

In [ ]:
import torch.nn.functional as F

def supcon_loss(features, labels, temperature=0.07):
    features = F.normalize(features, dim=1)
    
    similarity = torch.matmul(features, features.T) / temperature
    
    labels = labels.unsqueeze(1)
    mask = labels == labels.T
    
    exp_sim = torch.exp(similarity)
    
    log_prob = similarity - torch.log(exp_sim.sum(dim=1, keepdim=True))
    
    loss = - (mask * log_prob).sum(dim=1) / mask.sum(dim=1)
    
    return loss.mean()

## 7. Margin cross-entropy loss (primary classification term)

In [ ]:
import torch.nn.functional as F

def margin_loss(logits, labels, margin=0.3):
    one_hot = F.one_hot(labels, num_classes=logits.size(1)).float()
    
    # subtract margin from correct class
    adjusted_logits = logits - margin * one_hot
    
    return F.cross_entropy(adjusted_logits, labels)

## 8. Class-balanced weighting for the discrepancy term

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
class_counts = np.bincount(y)
total_samples = len(y)

class_priors = class_counts / total_samples   # πy
class_priors = torch.tensor(class_priors, dtype=torch.float32)

## 9. Sinusoidal annealing schedule alpha(t)
Controls how strongly the theta_g vs theta_t discrepancy loss (`LMORE`) is weighted over training.

In [ ]:
import math

def alpha(t, T, A=1.5):
    return A * math.sin(math.pi * t / T)

## 10. theta(t) diagnostics
Measures how much of the model's prediction for each class comes from the low-rank correction vs. the shared general weights.

In [ ]:
def per_class_theta_effect(model, loader, class_counts):
    import numpy as np
    
    model.eval()
    
    class_diff = {}

    with torch.no_grad():
        for xb, yb in loader:
            
            out_full = model(xb)
            out_general = model.forward_general(xb)

            diff = ((out_full - out_general) ** 2).mean(dim=1)

            for i in range(len(yb)):
                cls = yb[i].item()
                class_diff.setdefault(cls, []).append(diff[i].item())

    print("\n🔍 Per-class θt effect:\n")
    
    for cls in sorted(class_diff):
        avg_diff = np.mean(class_diff[cls])
        count = class_counts[cls]
        
        print(f"Class {cls} (count={count}): θt effect = {avg_diff:.4f}")

In [ ]:
def check_theta_t_effect(model, loader, class_counts):
    model.eval()
    
    diffs_majority = []
    diffs_minority = []
    
    # define threshold (you can adjust)
    threshold = np.median(class_counts)

    with torch.no_grad():
        for xb, yb in loader:
            
            out_full = model(xb)
            out_general = model.forward_general(xb)

            diff = ((out_full - out_general) ** 2).mean(dim=1)

            for i in range(len(yb)):
                cls = yb[i].item()
                
                if class_counts[cls] >= threshold:
                    diffs_majority.append(diff[i].item())
                else:
                    diffs_minority.append(diff[i].item())

    print("Avg θt effect on MAJORITY:", np.mean(diffs_majority))
    print("Avg θt effect on MINORITY:", np.mean(diffs_minority))

## 11. Training loop

In [ ]:
model = Model(X.shape[1], len(np.unique(y)))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

T = len(loader) * 10   # total iterations (epochs=10)
step = 0

for epoch in range(10):
    model.train()
    total_loss = 0

    for xb, yb in loader:
        optimizer.zero_grad()

        # --- Forward passes ---
        out_full = model(xb)                 # θg + θt
        out_general = model.forward_general(xb)  # θg only

        # --- Base loss ---
        base_loss = margin_loss(out_full, yb, margin=0.3)
        # --- LMORE (discrepancy loss) ---
        diff = (out_full - out_general) ** 2
        M = diff.mean(dim=1)   # per-sample

        # get πy for each sample
        weights = class_priors[yb]

        LMORE = (weights * M).mean()

        # --- Sinusoidal weighting ---
        alpha_t = alpha(step, T, A=1.0)

        # --- Total loss ---
        feat = model.layer1(xb)

        c_loss = supcon_loss(feat, yb)

        loss = base_loss + alpha_t * LMORE + 0.3 * c_loss

        probs = torch.softmax(out_full, dim=1)
        penalty = probs[:, 3].mean()

        loss += 0.1 * penalty
        # --- Backprop ---
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        step += 1

        # --- DEBUG PRINT (every few steps to avoid spam) ---
        if step % 100 == 0:
            print(f"Step {step}")
            print(f"Base Loss: {base_loss.item():.4f}")
            print(f"LMORE: {LMORE.item():.4f}")
            print(f"Alpha(t): {alpha_t:.4f}")
            print("-" * 30)

    print(f"Epoch {epoch}: Loss = {total_loss:.4f}")

## 12. Evaluation on the training distribution

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds))

In [ ]:
for name, param in model.named_parameters():
    if "A" in name or "B" in name:
        print(name, param.norm().item())

In [ ]:
check_theta_t_effect(model, loader, class_counts)

In [ ]:
per_class_theta_effect(model, loader, class_counts)

## 13. Confusion matrices (raw + normalized)

In [ ]:
# ================================
# 1. Imports
# ================================
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# ================================
# 2. Get predictions
# ================================
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in loader:   # your existing dataloader
        
        xb = xb.to(next(model.parameters()).device)
        yb = yb.to(next(model.parameters()).device)
        
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

# Convert to numpy
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ================================
# 3. Compute confusion matrix
# ================================
cm = confusion_matrix(all_labels, all_preds)

# ================================
# 4. Plot NORMAL confusion matrix
# ================================
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

# ================================
# 5. Plot NORMALIZED confusion matrix (VERY IMPORTANT)
# ================================
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8,6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix")

plt.show()

## 14. t-SNE of intermediate features
With a focus on the historically hardest classes.

In [ ]:
# =====================================
# 1. Imports
# =====================================
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# =====================================
# 2. Extract features from model
# =====================================
model.eval()

features = []
labels = []

device = next(model.parameters()).device

with torch.no_grad():
    for xb, yb in loader:
        xb = xb.to(device)

        # 👉 Extract intermediate features (IMPORTANT)
        feat = model.layer1(xb)   # use first layer output
        
        features.append(feat.cpu().numpy())
        labels.append(yb.numpy())

# Convert to numpy arrays
features = np.vstack(features)
labels = np.hstack(labels)

print("Feature shape:", features.shape)

# =====================================
# 3. Subsample (VERY IMPORTANT for speed)
# =====================================
num_samples = 3000   # you can change to 2000–5000

if len(features) > num_samples:
    idx = np.random.choice(len(features), num_samples, replace=False)
    features = features[idx]
    labels = labels[idx]

# =====================================
# 4. Run t-SNE
# =====================================
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    max_iter=1000,
    random_state=42
)

X_embedded = tsne.fit_transform(features)

# =====================================
# 5. Plot full t-SNE
# =====================================
plt.figure(figsize=(8,6))

scatter = plt.scatter(
    X_embedded[:, 0],
    X_embedded[:, 1],
    c=labels,
    cmap='tab10',
    s=10
)

plt.colorbar(scatter)
plt.title("t-SNE Visualization (All Classes)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")

plt.show()

# =====================================
# 6. Plot ONLY problematic classes (2,3,7)
# =====================================
mask = np.isin(labels, [2, 3, 7])

plt.figure(figsize=(8,6))

plt.scatter(
    X_embedded[mask, 0],
    X_embedded[mask, 1],
    c=labels[mask],
    cmap='tab10',
    s=10
)

plt.title("t-SNE (Classes 2, 3, 7)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")

plt.show()

## 15. Final evaluation on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

# Classification Report
print("Classification Report (TEST):")
print(classification_report(all_labels, all_preds, digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test)")
plt.show()